In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import xgboost
import networkx as nx
from sklearn.ensemble import IsolationForest
from imblearn.over_sampling import SMOTE
import sys
sys.path.append('D:/Projects/fraud-detection-shap-aws')
from utils.config_loader import load_config

print("pandas    :", pd.__version__)
print("xgboost   :", xgboost.__version__)
print("networkx  :", nx.__version__)
print("shap      :", shap.__version__)
print("\n All libraries ready")

pandas    : 3.0.3
xgboost   : 3.3.0
networkx  : 3.6.1
shap      : 0.52.0

✓ All libraries ready


In [9]:
#Load Config
config = load_config('D:/Projects/fraud-detection-shap-aws/configs/creditcard_config.json')
#Load Dataset
df = pd.read_csv(f"D:/Projects/fraud-detection-shap-aws/{config['file_path']}")

#Detect Fraud Stats
fraud_count = df[config['target_column']].sum()
legit_count = len(df) / fraud_count
fraud_rate = fraud_count / len(df)
scale_pos_weight = legit_count / fraud_count
contamination = fraud_rate

print(f"\nDataset : {config['dataset_name']}")
print(f"Shape : {df.shape}")
print(f"Fraud : {fraud_count} ({fraud_rate*100:.3f}%)")
print(f"Legit : {legit_count}")
print(f"scale_pos_weight : {scale_pos_weight:.4f}")
print(f"contamination : {contamination:.4f}")
print("\n Dataset Loaded Successfully")
df.head()

Config loaded: Credit Card Fraud

Dataset : Credit Card Fraud
Shape : (284807, 31)
Fraud : 492 (0.173%)
Legit : 578.8760162601626
scale_pos_weight : 1.1766
contamination : 0.0017

 Dataset Loaded Successfully


,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


In [24]:
# Switch dataset with ONE line change
config2 = load_config('D:/Projects/fraud-detection-shap-aws/configs/paysim_config.json')
df2 = pd.read_csv(f"D:/Projects/fraud-detection-shap-aws/{config2['file_path']}")

fraud2 = df2[config2['target_column']].sum()
rate2 = fraud2 / len(df2)

print(f"Dataset : {config2['dataset_name']}")
print(f"Shape : {df2.shape}")
print(f"Fraud : {fraud2} ({rate2*100:.3f}%)")
print(f"scale_pos_weight : {(len(df2)-fraud2)/fraud2:.1f}")
print("\n Same pipeline. Different dataset. Zero code changes.")

Config loaded: PaySim Mobile Money Fraud
Dataset : PaySim Mobile Money Fraud
Shape : (6362620, 11)
Fraud : 8213 (0.129%)
scale_pos_weight : 773.7

 Same pipeline. Different dataset. Zero code changes.
